### Preprocesamiento de Datos: Agrupación Espacial y Estandarización Temporal

Este primer bloque de código del cuaderno de preprocesamiento sienta las bases estructurales del conjunto de datos, acondicionando tanto la variable objetivo (etiquetas espaciales) como el sustrato visual (los vídeos en crudo) para que sean digeribles por el *pipeline* automatizado de las iteraciones posteriores. La ejecución se divide en dos grandes operaciones de normalización: una espacial y otra temporal.

En la primera fase, el algoritmo aborda la simplificación del espacio de decisión predictivo. El *dataset* original clasifica los disparos en una cuadrícula de alta resolución compuesta por 9 zonas distintas de la portería. Sin embargo, para evaluar la lectura biomecánica de la lateralidad, este nivel de detalle resulta excesivo e introduce ruido innecesario. Mediante la función de mapeo, el código colapsa esta cuadrícula tridimensional en un esquema unidimensional de tres columnas verticales (agrupando las zonas 1, 4 y 7 en la etiqueta 1; 2, 5 y 8 en la etiqueta 2; y 3, 6 y 9 en la etiqueta 3). 



De este modo, el problema matemático se reduce a predecir trayectorias macroscópicas: derecha, centro e izquierda. Esta nueva taxonomía se inyecta en el archivo maestro de etiquetas, garantizando que todos los modelos posteriores operen sobre el mismo marco de referencia espacial.

En la segunda fase, el sistema resuelve un problema clásico de visión artificial: la varianza en la longitud de las secuencias de entrada. Los modelos de *Deep Learning* requieren tensores de dimensiones fijas, pero los clips de vídeo originales poseen duraciones heterogéneas. Para solucionar esto, el *script* impone una estandarización temporal estricta, forzando a que todos y cada uno de los vídeos tengan exactamente una longitud de **64 fotogramas**. 

La lógica aplicada para alcanzar este estándar temporal no es arbitraria y prioriza la preservación del clímax biomecánico (el momento del impacto y el seguimiento):

* **Recorte (CUT):** Si un vídeo excede los 64 fotogramas, el algoritmo recorta los *frames* sobrantes del principio de la secuencia, asumiendo que los primeros instantes de la carrera inicial contienen menos información crítica que la fase final del gesto.
* **Relleno (PADDING):** Por el contrario, si la secuencia es demasiado corta, el sistema aplica un relleno temporal duplicando estáticamente el primer fotograma tantas veces como sea necesario hasta alcanzar el objetivo. 

Esta técnica de relleno asegura que el *framerate* y la sincronización de la acción final no se vean alterados ni acelerados artificialmente. Finalmente, las secuencias procesadas son recodificadas y exportadas como nuevos archivos MP4, constituyendo el corpus visual limpio y uniforme sobre el que trabajarán los modelos multimodales.

In [ ]:
import pandas as pd
import cv2
import numpy as np
import os

def map_shoot_zone(zone):
    """Mapea las 9 zonas a 3 etiquetas principales: Derecha (1), Centro (2), Izquierda (3)"""
    if zone in [1, 4, 7]:
        return 1
    elif zone in [2, 5, 8]:
        return 2
    elif zone in [3, 6, 9]:
        return 3
    else:
        return np.nan

base_path = r'C:\Users\david\OneDrive\Desktop\TFT_LLMPenalties2026'
csv_path = os.path.join(base_path, 'labeled_frames_cuttedvids.csv')
videos_folder = os.path.join(base_path, 'vid')
output_folder = os.path.join(base_path, 'modified_videos')
os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(csv_path)
df['shoot_zone_grouped'] = df['shoot_zone'].apply(map_shoot_zone)
df.to_csv(os.path.join(base_path, 'labeled_frames_grouped.csv'), index=False)

def process_video_frames(video_path, target_frames=64):
    """
    Extrae los frames del vídeo y aplica Padding o Cut.
    Devuelve los frames procesados y la acción realizada.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        
    cap.release()
    
    num_frames = len(frames)
    
    if num_frames == 0:
        print(f"Error: No se pudo leer el vídeo {video_path}")
        return [], "ERROR", 0

    if num_frames > target_frames:
        action = "CUT"
        processed_frames = frames[-target_frames:]
    elif num_frames < target_frames:
        action = "PADDING"
        padding_size = target_frames - num_frames
        padding_frames = [frames[0]] * padding_size
        processed_frames = padding_frames + frames
    else:
        action = "NINGUNA"
        processed_frames = frames
        
    return processed_frames, action, num_frames

unique_videos = df['vid_ID'].unique()

for vid_id in unique_videos: 
    video_file = os.path.join(videos_folder, f"{vid_id}.mp4")
    output_file = os.path.join(output_folder, f"{vid_id}.mp4")
    
    if os.path.exists(video_file):
        frames_finales, accion, frames_orig = process_video_frames(video_file, target_frames=64)
        
        if len(frames_finales) > 0:
            height, width, layers = frames_finales[0].shape
            
            fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
            out = cv2.VideoWriter(output_file, fourcc, 30.0, (width, height))
            
            for frame in frames_finales:
                out.write(frame)
            
            out.release()
            
            print(f"Video {vid_id}: Original={frames_orig} frames -> Action={accion} -> Final={len(frames_finales)} frames.")
    else:
        print(f"Aviso: No se encontró el archivo {video_file}")


Procesando vídeos...
Video v1_p1_diag: Original=90 frames -> Action=CUT -> Final=64 frames.
Video v1_p2_diag: Original=89 frames -> Action=CUT -> Final=64 frames.
Video v1_p3_diag: Original=94 frames -> Action=CUT -> Final=64 frames.
Video v1_p4_diag: Original=87 frames -> Action=CUT -> Final=64 frames.
Video v1_p5_diag: Original=75 frames -> Action=CUT -> Final=64 frames.
Video v1_p6_diag: Original=113 frames -> Action=CUT -> Final=64 frames.
Video v1_p7_diag: Original=112 frames -> Action=CUT -> Final=64 frames.
Video v1_p8_diag: Original=76 frames -> Action=CUT -> Final=64 frames.
Video v1_p9_diag: Original=93 frames -> Action=CUT -> Final=64 frames.
Video v1_p10_diag: Original=67 frames -> Action=CUT -> Final=64 frames.
Video v1_p11_diag: Original=103 frames -> Action=CUT -> Final=64 frames.
Video v1_p12_diag: Original=92 frames -> Action=CUT -> Final=64 frames.
Video v1_p13_diag: Original=104 frames -> Action=CUT -> Final=64 frames.
Video v1_p14_diag: Original=76 frames -> Action

### Análisis Estadístico y Justificación de la Ventana Temporal

Este segundo bloque del cuaderno de preprocesamiento no ejecuta transformaciones sobre los vídeos, sino que actúa como un módulo de auditoría empírica. Su propósito fundamental es generar la evidencia estadística y visual que justifica matemáticamente las decisiones heurísticas tomadas en la celda anterior; específicamente, la elección de 64 fotogramas como ventana temporal estándar y la estrategia de recortar las secuencias desde su inicio.

El algoritmo comienza construyendo un *dataframe* analítico (`df_stats`) que disecciona la anatomía temporal de cada lanzamiento en el *dataset*. Agrupando los datos por el identificador del vídeo, el código calcula la longitud total de cada secuencia y rastrea la ubicación exacta del clímax biomecánico: el momento del impacto de la bota con el balón (`kick_frame == 1`). A partir de este ancla temporal, el sistema segmenta el vídeo en dos métricas críticas: la cantidad de fotogramas de la carrera de aproximación previos al golpeo y los fotogramas de seguimiento y vuelo del balón posteriores al mismo.

Con esta radiografía temporal construida, el código despliega la librería *Seaborn* para generar tres visualizaciones clave que fundamentan el diseño experimental:

1. **Distribución de la Longitud Total:** El primer histograma mapea la duración de todos los vídeos y traza dos líneas de referencia: la media global del conjunto de datos y el límite estricto de 64 fotogramas. Esta gráfica permite demostrar visualmente que el valor de 64 no es un número arbitrario, sino un umbral estadísticamente óptimo que cubre la campana de Gauss de la distribución, minimizando la necesidad de aplicar rellenos (*padding*) excesivos en los vídeos cortos o mutilaciones severas en los largos.
2. **Distribución del Momento de Impacto:** El segundo histograma evalúa en qué fotograma exacto ocurre el golpeo. Este gráfico es vital para constatar que el evento biomecánico principal sucede en una franja predecible de la secuencia, lo que garantiza que la información más importante no se encuentra dispersa de forma aleatoria a lo largo de los vídeos.
3. **Preservación del Contexto Biomecánico:** La última gráfica (un diagrama de caja o *boxplot*) aborda la validación más crítica del preprocesamiento. Al simular el recorte inverso a 64 fotogramas, la variable `kept_runup_frames` calcula cuántos *frames* de la carrera inicial sobreviven a la poda. Esta visualización demuestra matemáticamente que, al recortar el exceso de vídeo desde el principio, el sistema elimina únicamente los pasos iniciales —que aportan poca señal direccional—, logrando conservar un contexto previo suficiente para leer la inercia del jugador y asegurando que la valiosa fase post-impacto quede siempre intacta.

Finalmente, el *script* vuelca por consola un resumen estadístico detallado y exporta las tres gráficas en formato PNG de alta resolución. Estos documentos gráficos y numéricos no solo aseguran la integridad de los datos antes de alimentar a los modelos de visión artificial, sino que proporcionan el rigor metodológico necesario para defender la viabilidad de la estandarización del *dataset* en la memoria del proyecto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'figure.titlesize': 16, 'axes.titlesize': 14})

base_path = r'C:\Users\david\OneDrive\Desktop\TFT_LLMPenalties2026'
csv_path = os.path.join(base_path, 'labeled_frames_cuttedvids.csv')
df = pd.read_csv(csv_path)

df_grouped = df.groupby('vid_ID').agg(
    total_frames=('frame_id', 'count')
).reset_index()

kick_frames = df[df['kick_frame'] == 1][['vid_ID', 'frame_id']].rename(columns={'frame_id': 'kick_frame_index'})

df_stats = pd.merge(df_grouped, kick_frames, on='vid_ID', how='left')

df_stats['frames_after_kick'] = df_stats['total_frames'] - df_stats['kick_frame_index'] - 1
df_stats['frames_before_kick'] = df_stats['kick_frame_index']

# GRÁFICA 1: Longitud de los Vídeos
plt.figure(figsize=(10, 6))
sns.histplot(df_stats['total_frames'], bins=30, kde=True, color='skyblue')

mean_frames = df_stats['total_frames'].mean()
plt.axvline(mean_frames, color='green', linestyle='dashed', linewidth=2, 
            label=f'Media ({mean_frames:.1f} frames)')
plt.axvline(64, color='red', linestyle='dashed', linewidth=2, 
            label='Punto de Corte (64 frames)')

plt.title('Distribución de la Longitud Total de los Vídeos de Penaltis')
plt.xlabel('Número Total de Frames por Vídeo')
plt.ylabel('Frecuencia (Número de Vídeos)')
plt.legend()
plt.tight_layout()
plt.savefig('distribucion_longitud_videos.png', dpi=300)
plt.close()

# GRÁFICA 2: Momento del Impacto
plt.figure(figsize=(10, 6))
sns.histplot(df_stats['kick_frame_index'].dropna(), bins=30, kde=True, color='salmon')

mean_kick = df_stats['kick_frame_index'].mean()
plt.axvline(mean_kick, color='green', linestyle='dashed', linewidth=2, 
            label=f'Media del Impacto (Frame {mean_kick:.1f})')

plt.title('Distribución del Momento de Impacto del Balón')
plt.xlabel('Índice del Frame del Golpeo (kick_frame = 1)')
plt.ylabel('Frecuencia (Número de Vídeos)')
plt.legend()
plt.tight_layout()
plt.savefig('distribucion_momento_impacto.png', dpi=300)
plt.close()

# GRÁFICA 3: Justificación del "Cut"
df_stats['kept_runup_frames'] = 64 - df_stats['frames_after_kick'] - 1
df_stats['kept_runup_frames'] = df_stats[['kept_runup_frames', 'frames_before_kick']].min(axis=1)

plt.figure(figsize=(8, 6))
sns.boxplot(y=df_stats['kept_runup_frames'], color='lightgreen', width=0.4)

plt.title('Contexto Biomecánico Conservado (Ventana de 64 frames)')
plt.ylabel('Frames de Carrera de Aproximación Conservados')
plt.tight_layout()
plt.savefig('frames_carrera_conservados.png', dpi=300)
plt.close()

print("=== ESTADÍSTICAS DEL DATASET ===")
print(f"Total de vídeos: {len(df_stats)}")
print(f"Longitud media: {df_stats['total_frames'].mean():.2f} frames (Desviación: {df_stats['total_frames'].std():.2f})")
print(f"Frame medio de impacto: {df_stats['kick_frame_index'].mean():.2f}")
print(f"Frames medios de vuelo post-impacto: {df_stats['frames_after_kick'].mean():.2f}")
print(f"Media de frames de carrera conservados al cortar a 64: {df_stats['kept_runup_frames'].mean():.2f}")
print("Gráficos generados correctamente y guardados como .png.")

=== ESTADÍSTICAS DEL DATASET ===
Total de vídeos: 641
Longitud media: 77.93 frames (Desviación: 31.52)
Frame medio de impacto: 67.93
Frames medios de vuelo post-impacto: 9.00
Media de frames de carrera conservados al cortar a 64: 48.72
Gráficos generados correctamente y guardados como .png.
